# Modeling HUPA0003's using an RNN, glucose are considered as the only input at each step t  

In addition, in this notebook we take an approach of the form sequence to sequence i.e. given a sequence we predict several steps ahead 

In [47]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [48]:
import sys
sys.path.append("..")
from scripts.window_regressor import Window_Regressor_Sequence_to_Sequence

In [49]:
data = pd.read_csv('../Data/Preprocessed/HUPA0003P.csv', sep=';')
data['time'] = pd.to_datetime(data['time'])

## Architecture 

- input_size = 1 (Size of vector at each time step)
- hidden_size = 10 (Number of recurrent neurons)
- batch_size = 12

In [50]:
import torch 
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader

In [51]:
batch_size = 24
input_size = 1
hidden_size = 12
window_size = 7
steps_squence_len = 6

In [52]:
train = Window_Regressor_Sequence_to_Sequence(time_series_data=data['glucose'][:-batch_size], window_size=window_size, horizon=steps_squence_len).generate_data_set()
test = Window_Regressor_Sequence_to_Sequence(time_series_data=data['glucose'][-batch_size:], window_size=window_size, horizon=steps_squence_len).generate_data_set()

In [53]:
train

,Predictor_0,Predictor_1,Predictor_2,Predictor_3,Predictor_4,Predictor_5,Predictor_6,X_t+1,X_t+2,X_t+3,X_t+4,X_t+5,X_t+6
0,137.666667,137.000000,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000
1,137.000000,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000
2,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667
3,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667,115.333333
4,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667,115.333333,114.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3729,100.333333,98.000000,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667
3730,98.000000,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000
3731,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000,92.333333
3732,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000,92.333333,97.666667


In [54]:
test

,Predictor_0,Predictor_1,Predictor_2,Predictor_3,Predictor_4,Predictor_5,Predictor_6,X_t+1,X_t+2,X_t+3,X_t+4,X_t+5,X_t+6
0,108.000000,113.000000,118.000000,124.666667,131.333333,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667
1,113.000000,118.000000,124.666667,131.333333,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333
2,118.000000,124.666667,131.333333,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000
3,124.666667,131.333333,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333
4,131.333333,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667
5,138.000000,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667,166.000000
6,135.333333,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667,166.000000,163.333333
7,132.666667,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667,166.000000,163.333333,160.666667
8,130.000000,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667,166.000000,163.333333,160.666667,158.000000
9,144.000000,158.000000,172.000000,172.666667,173.333333,174.000000,171.333333,168.666667,166.000000,163.333333,160.666667,158.000000,159.000000


In [55]:
train_data = torch.tensor(train.values, dtype=torch.float32)
test_data = torch.tensor(test.values, dtype=torch.float32)

In [56]:
train_dataset = TensorDataset(train_data)
test_dataset = TensorDataset(test_data)
train_loader = DataLoader(train_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

Define the model 

In [75]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, horizon_sequence):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, horizon_sequence)
    
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)  # h_n: (1, batch, hidden_size)
        out = self.linear(h_n.squeeze(0))  # (batch, horizon)
        return out

In [76]:
model = RNN(input_size=input_size, hidden_size=hidden_size, horizon_sequence=steps_squence_len)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()
epochs = 500

Train the model 

In [77]:
train_losses = []
for e in range(epochs):
    epoch_losses = []
    for x in train_loader:
        batch = x[0]                
        x_in = batch[:, :-steps_squence_len]        ## to extract predictors 
        y = batch[:, -steps_squence_len:]            # to etract label 
        x_in = x_in.unsqueeze(-1)  ## since torch receives an input (Batch,Window,Input_size=1) input size is equal to 1 cause we have an univariate time series 
        y_pred = model(x_in)        
        loss = loss_function(y_pred, y)
        epoch_losses.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    train_losses.append(avg_loss)
    if e % 10 == 0:
        print(f'Epoch {e}/{epochs}, Average Loss: {avg_loss:.4f}')

Epoch 0/500, Average Loss: 24221.4452
Epoch 10/500, Average Loss: 19296.1546
Epoch 20/500, Average Loss: 15399.1842
Epoch 30/500, Average Loss: 12351.8287
Epoch 40/500, Average Loss: 9899.6505
Epoch 50/500, Average Loss: 7975.6797
Epoch 60/500, Average Loss: 6456.3600
Epoch 70/500, Average Loss: 5109.1429
Epoch 80/500, Average Loss: 4136.5658
Epoch 90/500, Average Loss: 3380.0808
Epoch 100/500, Average Loss: 2785.0579
Epoch 110/500, Average Loss: 2313.4838
Epoch 120/500, Average Loss: 1935.3180
Epoch 130/500, Average Loss: 1630.5612
Epoch 140/500, Average Loss: 1384.2797
Epoch 150/500, Average Loss: 1185.0451
Epoch 160/500, Average Loss: 1023.7391
Epoch 170/500, Average Loss: 891.7761
Epoch 180/500, Average Loss: 783.4022
Epoch 190/500, Average Loss: 696.2785
Epoch 200/500, Average Loss: 625.6499
Epoch 210/500, Average Loss: 567.7830
Epoch 220/500, Average Loss: 520.2693
Epoch 230/500, Average Loss: 481.1449
Epoch 240/500, Average Loss: 448.5563
Epoch 250/500, Average Loss: 421.4599
Ep

Test the model

In [78]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [79]:
with torch.no_grad():
    for x in test_loader:      
        batch = x[0]        
        x_in = batch[:, :-steps_squence_len]        ## to extract predictors 
        y = batch[:, -steps_squence_len:]            # to etract label 
        x_in = x_in.unsqueeze(-1)  ## since torch receives an input (Batch,Window,Input_size=1) input size is equal to 1 cause we have an univariate time series 
        y_pred = model(x_in)  
        mse = mean_squared_error(y, y_pred)
        mae = mean_absolute_error(y, y_pred)
        mape = mean_absolute_percentage_error(y, y_pred)
        print(f'MAPE:{mape}')
        print(f'MAE:{mae}')
        print(f'MSE:{mse}')
        

MAPE:0.08990725129842758
MAE:15.018731117248535
MSE:430.0707702636719


Measure the behavior given the last window

In [68]:
train

,Predictor_0,Predictor_1,Predictor_2,Predictor_3,Predictor_4,Predictor_5,Predictor_6,X_t+1,X_t+2,X_t+3,X_t+4,X_t+5,X_t+6
0,137.666667,137.000000,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000
1,137.000000,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000
2,136.333333,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667
3,135.666667,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667,115.333333
4,135.000000,134.333333,133.666667,133.000000,128.000000,123.000000,118.000000,118.000000,118.000000,118.000000,116.666667,115.333333,114.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3729,100.333333,98.000000,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667
3730,98.000000,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000
3731,92.666667,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000,92.333333
3732,87.333333,82.000000,79.666667,77.333333,75.000000,74.666667,74.333333,74.000000,78.333333,82.666667,87.000000,92.333333,97.666667


In [80]:
with torch.no_grad():
    last_window = torch.tensor([137.666667, 137.000000, 136.333333, 135.666667, 135.000000, 134.333333, 133.666667])
    last_window = last_window.unsqueeze(0).unsqueeze(-1)
    y_pred = model(last_window)
    print(y_pred)


tensor([[131.4928, 131.6493, 132.1156, 132.4550, 132.6185, 132.6874]])
